In [1]:
import torch 
from torch import nn
from torch.nn import functional as F

In [2]:
data = [[[1, 2, 3],[4, 5, 6],[7, 8, 9]]]
my_tensor = torch.tensor(data)
print(my_tensor)

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])


In [3]:
shape = (2,3)

ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.rand(shape)
print(f"Random tensor: \n{random}\n")
print(f"Ones tensor: \n{ones}\n")
print(f"Zeros tensor: \n{zeros}\n")

print(shape)

Random tensor: 
tensor([[0.6648, 0.3443, 0.0356],
        [0.0810, 0.1792, 0.1225]])

Ones tensor: 
tensor([[1., 1., 1.],
        [1., 1., 1.]])

Zeros tensor: 
tensor([[0., 0., 0.],
        [0., 0., 0.]])

(2, 3)


In [4]:
#mimicking like given a tensor of shape (2,3) we need another tensor of shape same .

template = torch.tensor([[1,2], [3,4]]) 

#random_tensor = torch.rand_like(template)
random_tensor = torch.randn_like(template , dtype = torch.float32)
print(f"Random tensor :  \n {random_tensor}\n")

Random tensor :  
 tensor([[ 0.6848, -0.1808],
        [ 2.1827, -0.7699]])



In [5]:
# Shape  , type and device 
## float32 are important for neural networks as they are used to calculate the gradients and update the weights of the model during training for learning .

tensor = torch.tensor([[2, 3]] , dtype = torch.float32 )

print(f"Shape of tensor : {tensor.shape}")
print(f"Datatype of tensor : {tensor.dtype}")
print(f"Device tensor is stored on : {tensor.device}")

Shape of tensor : torch.Size([1, 2])
Datatype of tensor : torch.float32
Device tensor is stored on : cpu


#### Autograd = pytorch automatic deferentiation engine that powers neural network training . It records all the operations that happen on a tensor and allows us to calculate gradients automatically when we call .backward() on a tensor .

In [6]:
x = torch.tensor([[[1, 2, 3], [4, 5, 6]]])
w = torch.tensor([[[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]] , requires_grad = True)

In [7]:
print(f"X : \n {x.requires_grad}\n")
print(f"W : \n {w.requires_grad}\n")

X : 
 False

W : 
 True



In [8]:
a = torch.tensor(2.0 , requires_grad = True)
b = torch.tensor(3.0 , requires_grad = True)
x = torch.tensor(4.0 , requires_grad= True )

In [9]:
y = a + b 
z = y * x

print(f"z : {z} \n")
print(f"z : {z.grad_fn} \n")
print(f"y grad fn : {y.grad_fn} \n")
print(f"x grad fn : {x.grad_fn} \n")

z : 20.0 

z : <MulBackward0 object at 0x000002AE14D70910> 

y grad fn : <AddBackward0 object at 0x000002AE14D70910> 

x grad fn : None 



In [10]:
a = torch.tensor(  [[1,2] , [3,4] ] ) 
b = torch.tensor( [[5,6] , [7,8]] )
c = a* b 
print(f"c : {c} \n")
# element wise multiplication  [[5,12] , [21,32]]   




c : tensor([[ 5, 12],
        [21, 32]]) 



In [11]:
# Matrix multiplication
a = torch.tensor(  [[1,2] , [3,4] ] )
b = torch.tensor( [[5,6] , [7,8]] )
c = a @ b
print(f"c : {c} \n")

c : tensor([[19, 22],
        [43, 50]]) 



In [12]:
x = torch.arange(1, 10)
x = x.view(1, 3, 3)
print(f"x : \n {x}\n")

x : 
 tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])



In [13]:
data = torch.tensor([
    [10, 11, 12, 13],  # row 0
    [20, 21, 22, 23],  # row 1
    [30, 31, 32, 33]   # row 2
])

# Our "personalized list" of column indices to select from each row
indices_to_select = torch.tensor([[2], [0], [3]])

# Gather from `data` along dim=1 (the column dimension)
selected_values = torch.gather(data, dim=1, index=indices_to_select)

print(f"Selected Values:\n {selected_values}")

Selected Values:
 tensor([[12],
        [20],
        [33]])


## Implementing a Linear Regression

ŷ = XW + b

Where:

X is our input data.
W is the weight parameter.
b is the bias parameter.
ŷ is our model's prediction.

goal is to find the best W and b to make ŷ as close to the true y as possible.

In [14]:
"""
Linear Regression from Scratch in PyTorch (No nn.Module)
Target function: y = 2x + 1
"""

import torch

# ──────────────────────────────────────────────
# 1. CREATE TOY DATA  (y = 2x + 1 + noise)
# ──────────────────────────────────────────────
torch.manual_seed(42)

X = torch.linspace(0, 5, 50).unsqueeze(1)          # shape: (50, 1)
y_true = 2 * X + 1 + 0.3 * torch.randn_like(X)    # noisy targets

# ──────────────────────────────────────────────
# 2. INIT LEARNABLE PARAMETERS
# ──────────────────────────────────────────────
# These are the two numbers the model will learn:
#   w (weight) → should converge to ~2
#   b (bias)   → should converge to ~1

w = torch.randn(1, requires_grad=True)   # random start
b = torch.zeros(1, requires_grad=True)   # start at 0

# ──────────────────────────────────────────────
# 3. HYPERPARAMETERS
# ──────────────────────────────────────────────
lr = 0.01        # learning rate – how big a step we take
epochs = 200     # how many times we loop over the data

# ──────────────────────────────────────────────
# 4. TRAINING LOOP
# ──────────────────────────────────────────────
for epoch in range(epochs):

    # --- Forward pass: predict y ---
    y_pred = w * X + b                          # our linear model

    # --- Compute loss (Mean Squared Error) ---
    loss = ((y_pred - y_true) ** 2).mean()      # scalar

    # --- Backward pass: compute gradients ---
    loss.backward()          # fills w.grad and b.grad

    # --- Update parameters (gradient descent) ---
    with torch.no_grad():                       # don't track these ops
        w -= lr * w.grad
        b -= lr * b.grad

    # --- Zero gradients for next iteration ---
    w.grad.zero_()
    b.grad.zero_()

    # --- Print progress every 20 epochs ---
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>3d} | Loss: {loss.item():.4f} | "
              f"w = {w.item():.4f}, b = {b.item():.4f}")

# ──────────────────────────────────────────────
# 5. FINAL RESULT
# ──────────────────────────────────────────────
print("\n--- Training Complete ---")
print(f"Learned:  y = {w.item():.4f}·x + {b.item():.4f}")
print(f"Actual:   y = 2.0000·x + 1.0000")

Epoch  20 | Loss: 0.1397 | w = 2.0157, b = 0.8123


Epoch  40 | Loss: 0.1064 | w = 2.0547, b = 0.8449
Epoch  60 | Loss: 0.1048 | w = 2.0502, b = 0.8623
Epoch  80 | Loss: 0.1035 | w = 2.0454, b = 0.8779


Epoch 100 | Loss: 0.1023 | w = 2.0411, b = 0.8921
Epoch 120 | Loss: 0.1014 | w = 2.0372, b = 0.9051
Epoch 140 | Loss: 0.1007 | w = 2.0336, b = 0.9168
Epoch 160 | Loss: 0.1000 | w = 2.0303, b = 0.9275


Epoch 180 | Loss: 0.0995 | w = 2.0273, b = 0.9373
Epoch 200 | Loss: 0.0991 | w = 2.0246, b = 0.9461

--- Training Complete ---
Learned:  y = 2.0246·x + 0.9461
Actual:   y = 2.0000·x + 1.0000


In [15]:
# We'll create a "batch" of 10 data points
N = 10
# Each data point has 1 feature
D_in = 1
# The output for each data point is a single value
D_out = 1

# Create our input data X
# Shape: (10 rows, 1 column)
X = torch.randn(N, D_in)

# Create our true target labels y by applying the "true" function
# and adding some noise for realism
true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1 # Add a little noise

print(f"Input Data X (first 3 rows):\n {X[:3]}\n")
print(f"True Labels y_true (first 3 rows):\n {y_true[:3]}")

Input Data X (first 3 rows):
 tensor([[ 1.2580],
        [-1.5890],
        [-1.1208]])

True Labels y_true (first 3 rows):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])


In [16]:
# Initialize our parameters with random values
# Shapes must be correct for matrix multiplication: X(10,1) @ W(1,1) -> (10,1)
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Initial Weight W:\n {W}\n")
print(f"Initial Bias b:\n {b}")

Initial Weight W:
 tensor([[-0.7290]], requires_grad=True)

Initial Bias b:
 tensor([0.7277], requires_grad=True)


In [17]:
# Perform the forward pass to get our first prediction
y_hat = X @ W + b

print(f"Shape of our prediction y_hat: {y_hat.shape}\n")
print(f"Prediction y_hat (first 3 rows):\n {y_hat[:3]}\n")
print(f"True Labels y_true (first 3 rows):\n {y_true[:3]}")

Shape of our prediction y_hat: torch.Size([10, 1])



Prediction y_hat (first 3 rows):
 tensor([[-0.1894],
        [ 1.8860],
        [ 1.5447]], grad_fn=<SliceBackward0>)

True Labels y_true (first 3 rows):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])


In [18]:
# y_hat is our prediction from the forward pass
# y_true is the ground truth
# Let's calculate the loss manually
error = y_hat - y_true
squared_error = error ** 2
loss = squared_error.mean()

print(f"Prediction (first 3):\n {y_hat[:3]}\n")
print(f"Truth (first 3):\n {y_true[:3]}\n")
print(f"Loss (a single number): {loss}")

Prediction (first 3):
 tensor([[-0.1894],
        [ 1.8860],
        [ 1.5447]], grad_fn=<SliceBackward0>)

Truth (first 3):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])

Loss (a single number): 10.136190414428711


In [19]:
loss.backward()
# The gradients are now stored in the .grad attribute of our parameters
print(f"Gradient for W (∂L/∂W):\n {W.grad}\n")
print(f"Gradient for b (∂L/∂b):\n {b.grad}")

Gradient for W (∂L/∂W):
 tensor([[-7.2587]])

Gradient for b (∂L/∂b):
 tensor([-0.1442])


In [20]:
# Hyperparameters
learning_rate = 0.01
epochs = 100

# Let's re-initialize our random parameters
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Starting Parameters: W={W.item():.3f}, b={b.item():.3f}\n")

# The Training Loop
for epoch in range(epochs):
    ### STEP 1 & 2: Forward Pass and Loss Calculation ###
    y_hat = X @ W + b
    loss = torch.mean((y_hat - y_true)**2)

    ### STEP 3: Backward Pass (Calculate Gradients) ###
    loss.backward()

    ### STEP 4: Update Parameters (The Gradient Descent Step) ###
    # We wrap this in no_grad() because this is not part of the model's computation
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    ### STEP 5: Zero the Gradients ###
    # We must reset the gradients for the next iteration
    W.grad.zero_()
    b.grad.zero_()

    # Optional: Print progress
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

print(f"\nFinal Parameters: W={W.item():.3f}, b={b.item():.3f}")
print(f"True Parameters:  W=2.000, b=1.000")

Starting Parameters: W=-0.007, b=-0.978

Epoch 00: Loss=8.9257, W=0.044, b=-0.941
Epoch 10: Loss=5.7206, W=0.489, b=-0.609
Epoch 20: Loss=3.6841, W=0.834, b=-0.331
Epoch 30: Loss=2.3839, W=1.102, b=-0.100
Epoch 40: Loss=1.5499, W=1.310, b=0.092
Epoch 50: Loss=1.0126, W=1.473, b=0.251
Epoch 60: Loss=0.6651, W=1.599, b=0.384
Epoch 70: Loss=0.4393, W=1.698, b=0.493


Epoch 80: Loss=0.2922, W=1.774, b=0.584


Epoch 90: Loss=0.1960, W=1.835, b=0.660

Final Parameters: W=1.877, b=0.716
True Parameters:  W=2.000, b=1.000


In [21]:
# The input to our model has 1 feature (D_in=1)
# The output of our model is 1 value (D_out=1)
D_in = 1
D_out = 1

# Create a Linear layer
linear_layer = torch.nn.Linear(in_features=D_in, out_features=D_out)

# You can inspect the randomly initialized parameters inside
print(f"Layer's Weight (W): {linear_layer.weight}\n")
print(f"Layer's Bias (b): {linear_layer.bias}\n")

# You use it just like a function. Let's pass our data X through it.
# This performs the forward pass: X @ W.T + b
# (Note: nn.Linear stores W as (D_out, D_in), so it uses a transpose)
y_hat_nn = linear_layer(X)

print(f"Output of nn.Linear (first 3 rows):\n {y_hat_nn[:3]}")

Layer's Weight (W): Parameter containing:
tensor([[-0.4309]], requires_grad=True)

Layer's Bias (b): Parameter containing:
tensor([-0.5987], requires_grad=True)

Output of nn.Linear (first 3 rows):
 tensor([[-1.1407],
        [ 0.0860],
        [-0.1157]], grad_fn=<SliceBackward0>)


In [22]:
# Create a ReLU activation function layer
relu = torch.nn.ReLU()

# Let's create some sample data with positive and negative values
sample_data = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])

# Apply the activation
activated_data = relu(sample_data)

print(f"Original Data:      {sample_data}")
print(f"Data after ReLU:    {activated_data}")

Original Data:      tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
Data after ReLU:    tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])


In [23]:
# Create a GELU activation function layer
gelu = torch.nn.GELU()

# Use the same sample data
sample_data = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])

# Apply the activation
activated_data = gelu(sample_data)

print(f"Original Data:      {sample_data}")
print(f"Data after GELU:    {activated_data}")

Original Data:      tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
Data after GELU:    tensor([-0.0455, -0.1543,  0.0000,  0.3457,  1.9545])


In [24]:
# Softmax must be told which dimension contains the scores to be normalized.
# For a batch of predictions, this is typically the last dimension (dim=-1 or dim=1).
softmax = torch.nn.Softmax(dim=-1)

# Let's create some sample logits for a batch of 2 items, with 4 possible classes
# A higher number means the model is more confident in that class.
logits = torch.tensor([
    [1.0, 3.0, 0.5, 1.5],  # Logits for item 1
    [-1.0, 2.0, 1.0, 0.0]   # Logits for item 2
])

# Apply the softmax
probabilities = softmax(logits)

print(f"Original Logits:\n {logits}\n")
print(f"Output Probabilities:\n {probabilities}\n")
print(f"Sum of probabilities for item 1: {probabilities[0].sum()}")
print(f"Sum of probabilities for item 2: {probabilities[1].sum()}")

Original Logits:
 tensor([[ 1.0000,  3.0000,  0.5000,  1.5000],
        [-1.0000,  2.0000,  1.0000,  0.0000]])

Output Probabilities:
 tensor([[0.0939, 0.6942, 0.0570, 0.1549],
        [0.0321, 0.6439, 0.2369, 0.0871]])

Sum of probabilities for item 1: 1.0
Sum of probabilities for item 2: 1.0000001192092896


In [25]:
# --- nn.Embedding ---
# Let's define a small vocabulary and embedding size
vocab_size = 10       # We have 10 unique words in our language
embedding_dim = 3     # Each word will be represented by a vector of size 3

# Create the embedding layer
embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Input: A batch of tokenized sentences. Let's make a batch of 1 sentence with 4 words.
# The numbers are the integer IDs for each word in our vocabulary.
input_ids = torch.tensor([[1, 5, 0, 8]]) # Shape: (batch_size=1, sequence_length=4)

# Get the vectors for our sentence
word_vectors = embedding_layer(input_ids)

print(f"Embedding Layer's Weight Matrix (shape {embedding_layer.weight.shape}):\n {embedding_layer.weight.data}\n")
print(f"Input Token IDs (shape {input_ids.shape}):\n {input_ids}\n")
print(f"Output Word Vectors (shape {word_vectors.shape}):\n {word_vectors}")

Embedding Layer's Weight Matrix (shape torch.Size([10, 3])):
 tensor([[ 1.0311, -0.7048,  1.0131],
        [-0.3308,  0.5177,  0.3878],
        [-0.5797, -0.1691, -0.5733],
        [ 0.5069, -0.4752, -0.4920],
        [ 0.2704, -0.5628, -0.7328],
        [ 0.1043,  1.0414, -0.3997],
        [-2.2933,  0.4976, -0.4257],
        [-1.3371, -1.1955,  0.8123],
        [-0.3063, -0.3302, -0.9808],
        [ 0.1947, -1.6535,  0.6814]])

Input Token IDs (shape torch.Size([1, 4])):
 tensor([[1, 5, 0, 8]])

Output Word Vectors (shape torch.Size([1, 4, 3])):
 tensor([[[-0.3308,  0.5177,  0.3878],
         [ 0.1043,  1.0414, -0.3997],
         [ 1.0311, -0.7048,  1.0131],
         [-0.3063, -0.3302, -0.9808]]], grad_fn=<EmbeddingBackward0>)


In [26]:
# --- nn.LayerNorm ---
# LayerNorm needs to know the shape of the features it's normalizing.
# Our word vectors from the embedding example have a feature dimension of 3.
feature_dim = 3
norm_layer = torch.nn.LayerNorm(normalized_shape=feature_dim)

# Input: A batch of feature vectors. Let's create one.
input_features = torch.tensor([[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]]) # Shape (1, 2, 3)

# Apply the normalization
normalized_features = norm_layer(input_features)

# Let's check the mean and standard deviation of the output
output_mean = normalized_features.mean(dim=-1)
output_std = normalized_features.std(dim=-1)

print(f"Input Features:\n {input_features}\n")
print(f"Normalized Features:\n {normalized_features}\n")
print(f"Mean of each output vector (should be ~0): {output_mean}")
print(f"Std Dev of each output vector (should be ~1): {output_std}")

Input Features:
 tensor([[[1., 2., 3.],
         [4., 5., 6.]]])

Normalized Features:
 tensor([[[-1.2247,  0.0000,  1.2247],
         [-1.2247,  0.0000,  1.2247]]], grad_fn=<NativeLayerNormBackward0>)

Mean of each output vector (should be ~0): tensor([[0., 0.]], grad_fn=<MeanBackward1>)
Std Dev of each output vector (should be ~1): tensor([[1.2247, 1.2247]], grad_fn=<StdBackward0>)


In [27]:
# --- nn.Dropout ---
# Create a dropout layer that will zero out 50% of its inputs
dropout_layer = torch.nn.Dropout(p=0.5)

# Input: A simple tensor of ones so we can see the effect clearly.
input_tensor = torch.ones(1, 10) # Shape (1, 10)

# IMPORTANT: Dropout is only active during training. You must tell the layer
# it's in training mode with .train() or evaluation mode with .eval().
dropout_layer.train() # Activate dropout
output_during_train = dropout_layer(input_tensor)

dropout_layer.eval() # Deactivate dropout
output_during_eval = dropout_layer(input_tensor)

print(f"Input Tensor:\n {input_tensor}\n")
print(f"Output during training (randomly zeroed and scaled):\n {output_during_train}\n")
print(f"Output during evaluation (identity function):\n {output_during_eval}")

Input Tensor:
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])



Output during training (randomly zeroed and scaled):
 tensor([[2., 0., 0., 2., 0., 2., 2., 0., 0., 2.]])

Output during evaluation (identity function):
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


## Part 9: Professional PyTorch Workflow

Now that we have built the math manually, here is the same idea written in the standard PyTorch style. We will go in 5 steps: prepare data, define an `nn.Module`, create the optimizer and loss function, train the model, and then connect the idea to an LLM-style Feed-Forward Network.

In [28]:
# Step 1: Prepare a tiny regression dataset
torch.manual_seed(7)

X_model = torch.linspace(-1, 1, 100).unsqueeze(1)
y_model = 2 * X_model + 1 + 0.1 * torch.randn_like(X_model)

print(f"X_model shape: {X_model.shape}")
print(f"y_model shape: {y_model.shape}")
print("First 5 rows [x, y]:")
print(torch.cat((X_model[:5], y_model[:5]), dim=1))

X_model shape: torch.Size([100, 1])
y_model shape: torch.Size([100, 1])
First 5 rows [x, y]:
tensor([[-1.0000, -1.0820],
        [-0.9798, -0.9200],
        [-0.9596, -0.8293],
        [-0.9394, -1.0176],
        [-0.9192, -0.8551]])


In [29]:
# Step 2: Define the model blueprint with nn.Module
class LinearRegressionModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear_layer(x)


model = LinearRegressionModel(in_features=1, out_features=1)

print("Model Architecture:")
print(model)
print("\nModel Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

Model Architecture:
LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)

Model Parameters:
linear_layer.weight: tensor([[-0.5351]])
linear_layer.bias: tensor([-0.4700])


In [30]:
# Step 3: Create the optimizer and the loss function
import torch.optim as optim

learning_rate = 0.05
epochs = 100

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

print("Optimizer:", optimizer)
print("Loss Function:", loss_fn)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    weight_decay: 0
)
Loss Function: MSELoss()


In [31]:
# Step 4: Train with the clean PyTorch training loop
for epoch in range(epochs):
    y_hat = model(X_model)
    loss = loss_fn(y_hat, y_model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        w_learned = model.linear_layer.weight.item()
        b_learned = model.linear_layer.bias.item()
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={w_learned:.3f}, b={b_learned:.3f}")

print("\nFinal learned parameters:")
print(f"W={model.linear_layer.weight.item():.3f}")
print(f"b={model.linear_layer.bias.item():.3f}")

Epoch 00: Loss=4.2874, W=-0.485, b=-0.420
Epoch 10: Loss=2.3305, W=0.010, b=0.070
Epoch 20: Loss=1.0966, W=0.480, b=0.507
Epoch 30: Loss=0.4698, W=0.904, b=0.836
Epoch 40: Loss=0.2060, W=1.263, b=1.022
Epoch 50: Loss=0.0924, W=1.545, b=1.075
Epoch 60: Loss=0.0367, W=1.750, b=1.051
Epoch 70: Loss=0.0143, W=1.884, b=1.006
Epoch 80: Loss=0.0093, W=1.961, b=0.976
Epoch 90: Loss=0.0091, W=1.997, b=0.969

Final learned parameters:
W=2.008
b=0.974


In [32]:
# Step 5: Connect the idea to an LLM-style Feed-Forward Network (FFN)
class FeedForwardNetwork(nn.Module):
    def __init__(self, embedding_dim, ffn_dim):
        super().__init__()
        self.layer1 = nn.Linear(embedding_dim, ffn_dim)
        self.activation = nn.GELU()
        self.layer2 = nn.Linear(ffn_dim, embedding_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x


ffn = FeedForwardNetwork(embedding_dim=8, ffn_dim=16)
sample_input = torch.randn(2, 4, 8)
sample_output = ffn(sample_input)

print(ffn)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {sample_output.shape}")

FeedForwardNetwork(
  (layer1): Linear(in_features=8, out_features=16, bias=True)
  (activation): GELU(approximate='none')
  (layer2): Linear(in_features=16, out_features=8, bias=True)
)


Input shape:  torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])
